# NN Ensembler and Param Testbed

In [1]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings ('ignore', category = ConvergenceWarning)

In [2]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet_ensemble.csv"

# Preprocessing

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option ("display.max_columns", None)
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = \
[
    "Cattle_ID",
    "Farm_ID",
    "Feed_Quantity_lb",
    "Breed",
    "Climate_Zone",
    "Management_System",
    "Feed_Type",
    "Feeding_Frequency",
    "Walking_Distance_km",
    "Grazing_Duration_hrs",
    "Rumination_Time_hrs",
    "Resting_Hours",
    "Body_Condition_Score",
    "Humidity_percent",
    "BVD_Vaccine",
    "FMD_Vaccine",
    "Brucellosis_Vaccine",
    "HS_Vaccine",
    "BQ_Vaccine",
    "Housing_Score"
]
        
CATEGORICAL_FEATURES = \
[
    "Lactation_Stage",
    "Date",
    "Milking_Interval_hrs"
]

STANDARD_SCALED_FEATURES = \
[
    "Age_Months",
    "Weight_kg",
    "Parity",
    "Days_in_Milk",
    "Feed_Quantity_kg",
    "Water_Intake_L",
    "Ambient_Temperature_C",
    "Previous_Week_Avg_Yield"
]

In [4]:
def month_to_season (
    m
) -> str:
    """
    Converts month number into season string
    """
    if m in [12, 1, 2]:
        return "Winter"
    elif m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"


def preprocess (
    dtrain, dtest
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler]:
    """
    Further cleaning and feature engineering
    Returns: [dtrain, dtest, scaler]
    """
    # Drop useless features 
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    # Convert month to season
    months = pd.to_datetime (dtest['Date']).dt.month
    dtest = dtest.drop (columns = ['Date'])
    dtest['Date'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain = dtrain.drop (columns = ['Date'])
    dtrain['Date'] = months.apply (month_to_season)

    # Imputation
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    # One-hot encode categoricals
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, 
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, 
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    # Standardize numerics
    scaler = StandardScaler ()
    dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (dtrain[STANDARD_SCALED_FEATURES])
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

In [ ]:
# Feature engineer data
train_data = pd.read_csv (TRAIN_PATH)
X_train, X_test, y_train, y_test = train_test_split (
    train_data.drop (TARGET_FEATURE, axis = 1),
    train_data[TARGET_FEATURE], test_size = 0.2, random_state = 0)

X_train, X_test, scaler = preprocess (X_train, X_test)

: 

# Training Ensemble

In [ ]:
# Ensemble configs with solver-specific params
ensemble_configs = [{'hidden_layer_sizes': (100, 100, 100),
                     'alpha': 0},
                    
                    {'hidden_layer_sizes': (105, 105, 105),
                     'alpha': 0},

                    {'hidden_layer_sizes': (95, 95, 95),
                     'alpha': 0},

                    {'hidden_layer_sizes': (100, 100, 100), 
                        ,
                    
                    {'hidden_layer_sizes': (100, 100, 100), 
                     'alpha': 0.001, 
                     'activation': 'relu',
                     'solver': 'adam',
                     'solver_params': {'beta_1': 0.9,
                                       'beta_2': 0.999,
                                       'epsilon': 1e-8}},
                        
                    {'hidden_layer_sizes': (100, 100, 100),
                     'solver': 'sgd',
                     'momentum': 0.8,
                     'nesterovs_momentum': True,}]

# Train ensemble
models = []
train_rmse_list = []
test_rmse_list = []
best_test_rmse_list = []
best_iter_list = []

total_iterations = 250
iterations_per_step = 10

# Train all configs
print (f"Training ensemble of {len (ensemble_configs)} models...\n")
for idx, config in enumerate (ensemble_configs):
    print (f"Training model {idx+1}/{len (ensemble_configs)}")

    # Base common params
    base_params = dict (hidden_layer_sizes = config['hidden_layer_sizes'],
                        alpha              = config.get ('alpha', 0.0001),
                        solver             = config.get ('solver', 'adam'),
                        learning_rate_init = config.get ("learning_rate_init", 0.00003),
                        activation         = config.get ('activation', 'tanh'),
                        max_iter           = iterations_per_step,
                        learning_rate      = "adaptive",
                        random_state       = 0,
                        warm_start         = True,
                        early_stopping     = False,
                        n_iter_no_change   = 20,
                        verbose            = False)

    solver_params = config.get ('solver_params', {})
    all_params = {**base_params, **solver_params}

    model = MLPRegressor (**all_params)

    # Track best RMSE
    best_test_rmse = float ('inf')
    best_iter = 0

    for i in range (iterations_per_step, total_iterations + 1, iterations_per_step):
        model.fit (X_train, y_train)

        y_test_pred = model.predict (X_test)
        current_test_rmse = np.sqrt (mean_squared_error(y_test, y_test_pred))

        if current_test_rmse < best_test_rmse:
            best_test_rmse = current_test_rmse
            best_iter = i

    # Final evaluation
    y_train_pred = model.predict (X_train)
    y_test_pred = model.predict (X_test)
    train_rmse = np.sqrt (mean_squared_error (y_train, y_train_pred))
    test_rmse = np.sqrt (mean_squared_error (y_test, y_test_pred))

    print(f"  Final - Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")
    print(f"  Best test RMSE: {best_test_rmse:.4f} (at iteration {best_iter})")
    if test_rmse > best_test_rmse:
        print(f"  Overfitting detected: Test RMSE increased by {test_rmse - best_test_rmse:.4f}")
    print()

    models.append (model)
    train_rmse_list.append (train_rmse)
    test_rmse_list.append (test_rmse)
    best_test_rmse_list.append (best_test_rmse)
    best_iter_list.append (best_iter)

# Ensemble predictions
print ("=" * 60)
print ("ENSEMBLE RESULTS")
print ("=" * 60) 

train_preds = np.array ([model.predict (X_train) for model in models])
test_preds = np.array ([model.predict (X_test) for model in models])

ensemble_train_pred = np.median (train_preds, axis = 0)
ensemble_test_pred = np.median (test_preds, axis = 0)

ensemble_train_rmse = np.sqrt (mean_squared_error (y_train, ensemble_train_pred))
ensemble_test_rmse = np.sqrt (mean_squared_error (y_test, ensemble_test_pred))

print (f"\nIndividual model performance:")
print (f"  Final test RMSE - Best: {min (test_rmse_list):.4f}, Worst: {max (test_rmse_list):.4f}, Mean: {np.mean( test_rmse_list):.4f}")
print (f"  Best achieved test RMSE - Best: {min (best_test_rmse_list):.4f}, Worst: {max (best_test_rmse_list):.4f}, Mean: {np.mean (best_test_rmse_list):.4f}")
print (f"  Avg overfitting (final - best): {np.mean ([test_rmse_list[i] - best_test_rmse_list[i] for i in range (len (models))]):.4f}")

print (f"\nEnsemble performance:")
print (f"  Ensemble Train RMSE: {ensemble_train_rmse:.4f}")
print (f"  Ensemble Test RMSE:  {ensemble_test_rmse:.4f}")
print (f"  Improvement over best final individual: {min (test_rmse_list) - ensemble_test_rmse:.4f}")
print (f"  Improvement over best achieved individual: {min (best_test_rmse_list) - ensemble_test_rmse:.4f}")


Training ensemble of 6 models...

Training model 1/6
  Final - Train RMSE: 4.0965, Test RMSE: 4.1113
  Best test RMSE: 4.1107 (at iteration 200)
  Overfitting detected: Test RMSE increased by 0.0006

Training model 2/6
  Final - Train RMSE: 4.0960, Test RMSE: 4.1105
  Best test RMSE: 4.1100 (at iteration 200)
  Overfitting detected: Test RMSE increased by 0.0005

Training model 3/6
  Final - Train RMSE: 4.1005, Test RMSE: 4.1131
  Best test RMSE: 4.1131 (at iteration 250)

Training model 4/6


# Analyze Ensemble Predictions

In [ ]:
# Analyze best and worst predictions from ensemble
raw_data = pd.read_csv (TRAIN_PATH)

errors = np.sqrt ((y_test - ensemble_test_pred) ** 2)
df_results = X_test.copy ()
scaled_part = df_results[STANDARD_SCALED_FEATURES]
scaled_inverse = pd.DataFrame (scaler.inverse_transform (scaled_part),
                               columns = STANDARD_SCALED_FEATURES,
                               index = df_results.index)

# Replace only those columns
df_results[STANDARD_SCALED_FEATURES] = scaled_inverse
df_results["y_true"] = y_test
df_results["y_pred"] = ensemble_test_pred
df_results["rmse"] = errors

df_results = df_results.merge (raw_data,
                               left_index = True,
                               right_index = True,
                               how = "left")

print ("\nTop 5 BEST predictions:")
print (df_results.nsmallest (5, "rmse"))

print ("\nTop 5 WORST predictions:")
print (df_results.nlargest (5, "rmse"))


Top 5 BEST predictions:
        Age_Months_x  Weight_kg_x  Parity_x  Days_in_Milk_x  \
50397          125.0        302.4       1.0           277.0   
27280          128.0        473.7       3.0            39.0   
34167           49.0        354.8       3.0           275.0   
85585          141.0        367.2       6.0           118.0   
177669         143.0        342.9       2.0            15.0   

        Feed_Quantity_kg_x  Water_Intake_L_x  Ambient_Temperature_C_x  \
50397            14.706012         98.821862                34.921999   
27280            16.167715         73.562679                24.649142   
34167            13.525622         76.856935                32.172035   
85585            13.459435         77.728161                10.906007   
177669            8.556167         71.544850                29.635185   

        Anthrax_Vaccine_x  IBR_Vaccine_x  Rabies_Vaccine_x  \
50397                   1              1                 1   
27280                   0        

# Final Model - Train on Full Dataset

In [ ]:
# Build final ensemble on full training data
train_data = pd.read_csv (TRAIN_PATH)
test_data = pd.read_csv (TEST_PATH)

X_train_full = train_data.drop (TARGET_FEATURE, axis = 1)
y_train_full = train_data[TARGET_FEATURE]
X_test_full = test_data

X_train_full, X_test_full, scaler_full = preprocess (X_train_full, X_test_full)

print (f"Training final ensemble on full dataset...")
print ()

final_models = []

for idx, config in enumerate (ensemble_configs):
    print (f"Training final model {idx+1}/{len (ensemble_configs)}")
    
    base_params = dict (hidden_layer_sizes = config['hidden_layer_sizes'],
                        alpha              = config.get ('alpha', 0.0001),
                        solver             = config.get ('solver', 'adam'),
                        learning_rate_init = config.get ("learning_rate_init", 0.00003),
                        activation         = config.get ('activation', 'tanh'),
                        max_iter           = iterations_per_step,
                        learning_rate      = "adaptive",
                        random_state       = 0,
                        warm_start         = True,
                        early_stopping     = False,
                        n_iter_no_change   = 20,
                        verbose            = False)

    solver_params = config.get ('solver_params', {})
    all_params = {**base_params, **solver_params}
    model = MLPRegressor (**all_params)

    model.fit (X_train_full, y_train_full)
    final_models.append (model)
    print (f"  Model {idx+1} trained successfully")

print ("\nAll models trained!")

Training final ensemble on full dataset...

Training final model 1/2


KeyError: 'random_state'

In [ ]:
# Final Predictions - ensemble average
final_preds = np.array([model.predict(X_test_full) for model in final_models])
y_pred_final = final_preds.mean(axis=0)

print(f"Ensemble prediction mean: {y_pred_final.mean():.4f}")
print(f"Ensemble prediction std: {y_pred_final.std():.4f}")
print(f"Prediction range: [{y_pred_final.min():.4f}, {y_pred_final.max():.4f}]")

out_data = pd.DataFrame({
    'Cattle_ID': np.arange(1, len(y_pred_final) + 1),
    'Milk_Yield_L': y_pred_final
})
out_data.to_csv(OUT_PATH, index=False)
print(f"\nPredictions saved to {OUT_PATH}")

Ensemble prediction mean: 15.5881
Ensemble prediction std: 3.4514
Prediction range: [4.9853, 27.6118]

Predictions saved to ./data/out/nnet_ensemble.csv
